# ലെഷൻ 16 - Microsoft Foundry ഉപയോഗിച്ച് സ്‌കെയിലബിൾ ഏജന്റുകളെ വിന്യസിക്കൽ

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ‌ **Contoso** എന്ന കാവ്യകമ്പനിക്കായി ഒരു **ഉല്പാദന-സജ്ജകസ്റ്റമർ സപ്പോർട്ട് ഏജന്റ്** നിർമ്മിക്കും. മുമ്പത്തെ പാഠഭാഗങ്ങളിലേതിനേക്കാൾ വ്യത്യസ്തമായി, ഇവിടെ മുൾകേന്തരമായി ഏജന്റിന്റെ കാര്യക്ഷമ ചക്രം അല്ല — അതിൽ ചുറ്റിപ്പുറവും ചേർന്നിരിക്കുന്ന എല്ലാ ഘടകങ്ങളും ഏജന്റിനെ സ്‌കെയിലിൽ സുരക്ഷിതമായി പ്രവർത്തിപ്പിക്കാൻ സഹായിക്കുന്നവയാണ്:

1. **ടൂൾ കോളിംഗ്** — ഓർഡർ അന്വേഷിക്കല്‍ എന്നും ടിക്കറ്റ് സൃഷ്ടിയും.
2. **RAG** — ഒരു നോളedgeബേസിൽ നിന്ന് നയം ഉത്തരങ്ങൾ.
3. **മെമ്മറി** — സംവാദത്തിനിടയിൽ കസ്റ്റമറെ ഓർക്കുക.
4. **മോഡ്‌ൽ റൂട്ടിംഗ്** — ലഘുവായ അഭ്യർത്ഥനകൾ ചെറിയ മോഡലിന്, സങ്കീർണ്ണമായവ വലുതായ മോഡലിന് അയയ്ക്കുക.
5. **പ്രതികരണ കാഷിംഗ്** — മടങ്ങി വരുന്ന ചോദ്യങ്ങൾക്ക് മോഡൽ കോൾ വേണ്ടാതെ സേവനം നൽകുക.
6. **മനുഷ്യാനുമതി** — നിശ്ചിത പരിധിക്കുമുകളി ലാഭം ഉള്ള നിക്ഷേപങ്ങൾക്ക് അംഗീകാരം ലഭിക്കുന്ന വരെ നിർത്തുന്നു.
7. **മേൽനോട്ട ഗേറ്റ്** — മോശം റിലീസ് തടയാൻ ഓഫ്ലൈൻ പരിശോധന സെറ്റ്.
8. **ഓബ്സർവബിൾ** — ഓരോ അഭ്യർത്ഥനയിലും OpenTelemetry ട്രേസിംഗ്.

ഓരോ വിഭാഗവും സ്വതന്ത്രവും പ്രവർത്തനയോഗ്യവും ആണ്. ഓരോ വരിയും വായിക്കുക — ഉല്പാദന സവിശേഷതകൾ ജാഗ്രതയോടെ ചെറുതായി സൂക്ഷിച്ചിട്ടുണ്ട്.


## ക്രമീകരണം

ഈ നോട്ട്ബുക്ക് 실행 ചെയ്യുന്നതിന് മുൻപ്, നിങ്ങൾക്കുള്ളതെന്ന് ഉറപ്പാക്കുക:

1. **ഒരു Microsoft Foundry പ്രോജക്ട്** ഒരു ഡിപ്ലോയ് ചെയ്ത ചാറ്റ് മോഡൽ (ഉദാ., `gpt-4.1-mini`) ഉള്ളത്.
2. **Azure CLI-യിൽ ലോഗിൻ ചെയ്തിരിക്കുക** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` 실행 ചെയ്യുക.
3. **ആവശ്യമായ പരിസ്ഥിതി ചാരങ്ങൾ സജ്ജീകരിക്കുക:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്ട് എന്റ്പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങളുടെ ഡിപ്ലോയ്ഡ് മോഡലിന്റെ പേര്.

RAG വിഭാഗം ഉപയോഗിക്കുന്നു **Azure AI Search**, `AZURE_SEARCH_SERVICE_ENDPOINT` এবং `AZURE_SEARCH_API_KEY` സജ്ജീകരിച്ചിരിക്കുന്നപ്പോള്‍, അല്ലെങ്കില്‍ ഓര്‍മയിലുള്ള സെർച്ച് ഉപയോഗിച്ച് പിന്നോട്ടു പോകും, അതിനാൽ ഈ നോട്ട്ബുക്ക് സെർച്ച് റിസോഴ്‌സ് ഇല്ലാതെയും പ്രവർത്തിക്കും.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ഉപകരണങ്ങൾ

ഉത്പാദന ഉപകരണങ്ങൾ യഥാർത്ഥ സിസ്റ്റങ്ങൾക്കു πραγματική ജോലി ചെയ്യുന്നു. ഇവിടെ നാം ഒരു ഓർഡർ ഡാറ്റാബേസ്, ടിക്കറ്റിംഗ് സംവിധാനം സംവേദനം പൈതൺ ഫങ്ഷനുകളിലൂടെ സിമുലേറ്റ് ചെയ്യുന്നു. `@tool` ഡെക്കറേറ്റർ അവയെ ഏജന്റിലേക്ക് തുറന്നുവെക്കുന്നു.

`issue_refund` ഒരു ചിതറലിനും മുകളിലുള്ള മൂന്ന് ടിക്കറ്റുകൾക്കായുള്ള `approval_mode="always_require"` ഉപയോഗിക്കുന്നത് ശ്രദ്ധിക്കുക — ഇതാണ് പിന്നീട് നാം വിന്യസിക്കുന്ന മനുഷ്യൻ-ഇൻ-ദി-ലൂപ് പ്രാഥമികം.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — നയ വിജ്ഞാന അടിസ്ഥാനകോശം

നയത്തെക്കുറിച്ചുള്ള ചോദ്യങ്ങൾ ("നിങ്ങളുടെ റിട്ടേൺ വിൻഡോ എന്താണ്?") മോഡലിന്റെ സ്മരണയിലുള്ളത് ആണ് മറുപടി നൽകേണ്ടത് എന്നല്ല, അധികാരപ്രാപ്‌തമായ ഒരു ഉറവിടത്തിൽ നിന്നുള്ളവ ആയിരിക്കണം. നമുക്ക് ചെറിയ ഒരു വിജ്ഞാനകോശം ഒരു തിരയൽ ഉപകരണമായി ഉള്‍ക്കൂടുന്നു.

പ്രൊഡക്ഷനിൽ ഇത് **Azure AI Search** ആണ്; ഇവിടെ നോട്ട്ബുക്ക് എവിടെയെങ്കിലും പ്രവർത്തിക്കുമെന്ന് ഉറപ്പ് വരുത്താൻ ഇൻ-മെമ്മറി കീവർഡ് തിരയൽ നൽകുന്നു, പരിസ്ഥിതി വ്യത്യാസങ്ങൾ ഉണ്ടായപ്പോൾ സ്വയം Azure AI Search ലേക്ക് മാറുന്നു.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. മെമ്മറി

ആരെപ്പറ്റി സംസാരിക്കുന്നതാണെന്ന് മറന്നുകിട്ടുന്ന ഒരു പിന്തുണ ഏജന്റ് മോശം പിന്തുണ ഏജന്റാണ്. ഓരോ ഉപയോക്താവിനുമുള്ള ചെറിയ പ്രൊഫൈൽ സ്റ്റോർ ഞങ്ങൾ സൂക്ഷിക്കുകയും ഏജന്റിന്റെ നിർദ്ദേശങ്ങളിൽ ഒരു ചെറിയ സംഗ്രഹം ഉൾപ്പെടുത്തുകയും ചെയ്യുന്നു. ഉത്പാദനത്തിൽ ഇത് ഒരു മെമ്മറി സേവനമാണ് (പാഠം 13 കാണുക); ഇവിടെ ഒരു dict പാറ്റേൺ ശ്രദ്ധയാകർഷിക്കുന്നു.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. മോഡൽ റൂട്ടിംഗ് மற்றும் പ്രതികരണം കാഷിംഗ്

രണ്ട് ചിലവു നിയന്ത്രണങ്ങൾ ഒരൊറ്റ അഭ്യര്‍ത്ഥന കൈകാര്യമയത്തിലേക്ക് ബന്ധിപ്പിച്ചിരിക്കുന്നു:

- **റൂട്ടിംഗ്**: ഒരു വിലക്കേയുള്ള ഹ്യൂറിസ്റ്റിക് ക്ലാസിഫയറാണ് അഭ്യര്‍ത്ഥനയ്ക്ക് ചെറുതോ വലിയ മോഡലോ ആവശ്യമാണ് എന്ന് തീരുമാനിക്കുന്നത്.
- **കാഷിംഗ്**: സാധാരണ ആവർത്തിക്കുന്ന ചോദ്യങ്ങൾ മോഡൽ കോൾ ഇല്ലാതെ നേരിട്ട് ഒരു കാഷിൽ നിന്ന് സേർവ് ചെയ്യപ്പെടുന്നു.

ഇവിടെ ക്ലാസിഫയർ അവസരപരമായി ലളിതമാണ്. നിർമ്മാണത്തിൽ ഇത് ട്രാഫിക്കിനെതിരെ സാധൂകരിച്ച് Foundry-യുടെ മോഡൽ റൗട്ടറോടെ മാറ്റാം.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. ഏജന്റ്, മനുഷ്യ അംഗീകാരം, അതിന്റെ നിരീക്ഷണയോഗ്യത

ഇപ്പോൾ മേൽപ്പറഞ്ഞ ഉപകരണങ്ങളിൽ നിന്ന് ഏജന്റിനെ സംയോജിപ്പിച്ച് ഓരോ അഭ്യർത്ഥനയും OpenTelemetry സ്പാൻ എന്നതിൽ പൊതിഞ്ഞിരിക്കുന്നു. `handle_support_request` ഫംഗ്ഷൻ പ്രൊഡക്ഷൻ അഭ്യർത്ഥന കൈകാര്യം ചെയ്യുന്നതാണ്: cache → route → trace → run → cache.

മനുഷ്യ അംഗീകാരം ഫ്രെയിംവർക്കിൽനിന്നാണ് കൈകാര്യം ചെയ്യുന്നത്: `issue_refund`ന്റെ `approval_mode="always_require"` ആയതിനാൽ, റൺ താൽക്കാലികമായി നിർത്തിപ്പറുത്ത് നമ്മൾ വ്യക്തമായി തീർപ്പാക്കേണ്ട അംഗീകാരണ അഭ്യർത്ഥനയെ പുറംകാണിക്കുന്നു.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. മೌಲ്യനിർണ്ണയ ഗേറ്റ്

പാഠമുണ്ടാക്കുന്ന റിലീസ് ഗേറ്റാണ് ഇത്: ഒരു ഓഫ്ലൈൻ ടെസ്റ്റ് സെറ്റ് ഏജენტിനെ സ്കോർ ചെയ്യുന്നു, മറ്റു പല ഘട്ടങ്ങളേക്കാൾ തികഞ്ഞ പാസ്സ് നിരക്കുള്ളപ്പോഴേ നടത്തിപ്പിന് അനുമതി നൽകിയിട്ടുള്ളൂ. ഇവിടെ സ്കോറർ ഒരു ലളിതമായ കീവർഡ്-ഓവർലാപ്പ് പരിശോധനയാണ്, നോട്ട്‌ബുക്ക് സ്വയംഭരണമായി ഉണ്ടാക്കാൻ; ഉല്പാദനത്തിൽ, നിങ്ങൾ LLM-അസ്സ്-ജഡ്ജ് അല്ലെങ്കിൽ ഒരു ഫ്രെയിംവർക്കEvaluator (പാഠം 10 കാണുക) ഉപയോഗിക്കും.


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ഒന്നായി ചേർക്കുന്നു: ഒരു സിമുലേറ്റഡ് റിലീസ്

താഴെയുള്ള സെൽ പാഠം വിശദീകരിക്കുന്ന മുഴുവൻ ലൂപ് കാണിക്കുന്നു: മൂല്യനിർണയ ഗേറ്റ് ഓടിക്കുക, അത് കടന്നുപോവുന്ന പക്ഷം മാത്രം "ഡിപ്പ്ലോയ" ചെയ്യുക. ഒരു ഏജന്റ് പതിപ്പ് Foundry Agent Service ന് പ്രോത്സാഹിപ്പിക്കുന്നതിന് മുമ്പ് CI ൽ നിങ്ങൾ ഓടിക്കുന്ന പാറ്റേണാണ് ഇത്.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## സംക്ഷিপ്തം

നിങ്ങൾ എല്ലാ പ്രവർത്തനപരമായ പരിഗണനകളും ഒപ്പം ചേർത്തുകൊണ്ട് ഉൽപ്പന്നത്തിനായി തയ്യാറായ ഒരു കസ്റ്റമർ സപ്പോർട്ട് ഏജൻറ് സജ്ജമാക്കി:

- **സാധനങ്ങൾ, RAG, മെമ്മറി** ഏജന്റിനയ്ക്ക് കഴിവും പശ്ചാത്തലവും നൽകുന്നു.
- **മോഡൽ റൂട്ടിംഗ്, ക്യാഷിംഗ്** വൈകിയും ചെലവും നിയന്ത്രണത്തിൽ നിർത്തുന്നു.
- **മാനവ അംഗീകാരം** വലിയ റിഫണ്ട് പോലുള്ള ഉയർന്ന അപകടസാധ്യതയുള്ള പ്രവർത്തനങ്ങളെ സംരക്ഷിക്കുന്നു.
- **മൂല്യനിർണ്ണയ ഗേറ്റ്** മോശം റിലീസ് നടത്തുന്നതിനു മുൻപ് തടയുന്നു.
- **ട്രേസിംഗ്** എല്ലാ അഭ്യർത്ഥനകളും നിരീക്ഷണയോഗ്യമാക്കുന്നു.

### വെല്ലുവിളി

ഈ ഏജന്റിനെ വിപുലീകരിക്കുക:

1. **മൾട്ടി മോഡലുകൾ പിന്തുണയ്ക്കുക** — മൂന്നാം "കാരണമനസ്സിലാക്കൽ" പടി ചേർത്ത് എസ്കലേഷനുകളും പരാതി പരിഹാരങ്ങളും അതിലേക്ക് റൂട്ടുചെയ്യുക.
2. **മൂല്യനിർണ്ണയ ഗേറ്റുകൾ ചേർക്കുക** — `TEST_CASES` ൽ റിഫണ്ട് അംഗീകാര സീനാറിയോകൾ ഉൾപ്പെടുത്തട്ടെ, ഗേറ്റ് റിഗ്രഷനുകൾ പിടിക്കുന്നത് ഉറപ്പാക്കുക.
3. **ചെലവ് അറിവുള്ള റൂട്ടിംഗ് ചേർക്കുക** — ഓരോ അഭ്യർത്ഥനയുടെയും (ചെറിയത്, വലിയത്, ക്യാഷ്) അനുമാനിച്ച ചെലവ് ട്രാക്ക് ചെയ്ത് മിശ്രമായ ചോദനകൾ_BATCH_ ശേഷം ചെലവ് റിപ്പോർട്ട് അച്ചടിക്കുക.

അടുത്ത പാഠത്തിൽ, നിങ്ങൾ എതിര്‍പ്പുള്ള യാത്ര നടത്തുകയും Microsoft Foundry Local, Qwen ഉപയോഗിച്ച് നിങ്ങളുടെ സ്വന്തം യന്ത്രത്തിൽ ഒരൊറ്റേ ഏജന്റ് ഓടിക്കുകയും ചെയ്യും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
